In [1]:
# loading libraries
import numpy as np

import matplotlib.pyplot as plt 
import seaborn as sns

import pandas as pd

In [2]:
# import functions
import coriolis_functions 
# import selfbuild module
import coriolis_module
# check out if import worked as expected
print(dir(coriolis_module))
print(dir(coriolis_functions))

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_drift_distances', 'calculate_velocities', 'coriolis_acc', 'direction_vector', 'radius_at_latitude', 'rotation_matrix']
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_drift_distances', 'calculate_total_drift', 'calculate_velocities', 'coriolis_acc', 'haversine', 'main', 'np', 'radius_at_latitude', 'rotation_matrix']


In [3]:
# loading dataframe
column_names = ["FL_DATE", "ORIGIN", "DEST", "CANCELLED", "DIVERTED", "WHEELS_OFF", "WHEELS_ON"]
other_column_names = ["IATA", "LATITUDE", "LONGITUDE"]

fdf = pd.read_csv("data/flights_sample_3m.csv", usecols=column_names, na_filter=False)
ldf = pd.read_csv("data/airports.csv", usecols=other_column_names, na_filter=False)

fdf.dropna()
ldf.dropna()

print(len(fdf), " : number of flights in dataframe")
print(len(ldf), " : number of airport-locations in dataframe")

3000000  : number of flights in dataframe
341  : number of airport-locations in dataframe


In [4]:
boolean_columns = ["CANCELLED", "DIVERTED"]
fdf[boolean_columns] = fdf[boolean_columns].astype(bool)

In [5]:
fdf = fdf[~fdf["CANCELLED"]]

In [6]:
# checking unique values in both datasets
fdf_airports = set(fdf["ORIGIN"].unique()).union(set(fdf["DEST"].unique()))
ldf_airports = set(ldf["IATA"].unique())  

# finding missing airport codes in ldf
missing_airports = fdf_airports.difference(ldf_airports)

# dropping rows where "ORIGIN" or "DEST" are in missing_airports
fdf = fdf[~fdf["ORIGIN"].isin(missing_airports) & ~fdf["DEST"].isin(missing_airports)]

# merging fdf with ldf to add geographical data for ORIGIN and DEST
fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="ORIGIN", right_on="IATA", how="left")

fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="DEST", right_on="IATA", how="left", suffixes=("_ORIGIN", "_DEST"))

# dropping original origin and destination columns
fdf = fdf.drop(["ORIGIN", "DEST"], axis=1)

# enforcing categories on newly generated columns
fdf[["IATA_ORIGIN", "IATA_DEST"]] = fdf[["IATA_ORIGIN", "IATA_DEST"]].astype("category")

In [7]:
# checking dtypes, lost rows
print(fdf.dtypes, f"\n")
print(f"{len(fdf):,.2f},  : number of flights in dataframe\n")
fdf.head(10)

WHEELS_OFF            object
WHEELS_ON             object
CANCELLED               bool
DIVERTED                bool
IATA_ORIGIN         category
LATITUDE_ORIGIN      float64
LONGITUDE_ORIGIN     float64
IATA_DEST           category
LATITUDE_DEST        float64
LONGITUDE_DEST       float64
dtype: object 

2,870,784.00,  : number of flights in dataframe



,WHEELS_OFF,WHEELS_ON,CANCELLED,DIVERTED,IATA_ORIGIN,LATITUDE_ORIGIN,LONGITUDE_ORIGIN,IATA_DEST,LATITUDE_DEST,LONGITUDE_DEST
0,1210.0,1443.0,False,False,FLL,26.072583,-80.152750,EWR,40.692497,-74.168661
1,2123.0,2232.0,False,False,MSP,44.880547,-93.216922,SEA,47.448982,-122.309313
2,1020.0,1247.0,False,False,DEN,39.858408,-104.667002,MSP,44.880547,-93.216922
3,1635.0,1844.0,False,False,MSP,44.880547,-93.216922,SFO,37.619002,-122.374843
4,1853.0,2026.0,False,False,MCO,28.428889,-81.316028,DFW,32.895951,-97.037200
5,1252.0,1328.0,False,False,DAL,32.847114,-96.851772,OKC,35.393088,-97.600734
6,1024.0,1122.0,False,False,DCA,38.852083,-77.037722,BOS,42.364348,-71.005179
7,1659.0,1927.0,False,False,HSV,34.640448,-86.773109,DCA,38.852083,-77.037722
8,538.0,658.0,False,False,IAH,29.980472,-95.339722,LAX,33.942536,-118.408074
9,2135.0,2353.0,False,False,SEA,47.448982,-122.309313,FAI,64.813677,-147.859669


In [ ]:
# convert to datetime
fdf["FL_DATE"] = pd.to_datetime(fdf["FL_DATE"])

In [ ]:
time_columns = ["WHEELS_OFF", "WHEELS_ON"]

In [20]:
# helperfunction
def clean_time_format(value):
    try:
        # Attempt to convert the value to float
        time_value = float(value)
        hour = int(time_value // 100)  # Get the hour part
        minute = int(time_value % 100)  # Get the minute part
        return f"{hour:02}:{minute:02}"  # Format as HH:MM
    except (ValueError, TypeError):  # Handle non-convertible values
        return None  # Return None for invalid entries

In [21]:
# applying the cleaning function to the column
fdf[time_columns] = fdf[time_columns].apply(clean_time_format)

/tmp/ipykernel_68326/3759142872.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fdf[time_columns] = fdf[time_columns].apply(clean_time_format)


In [23]:
# dropping rows with invalid entries
fdf.dropna(subset=time_columns, inplace=True)

In [28]:
fdf["woff_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_OFF"])
fdf["won_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_ON"])

# dropping original time columns
fdf = fdf.drop(time_columns, axis=1)
fdf = fdf.drop(["FL_DATE"], axis=1)

ValueError: to assemble mappings requires at least that [year, month, day] be specified: [day,month,year] is missing

In [17]:
fdf.dtypes

WHEELS_OFF           float64
WHEELS_ON             object
CANCELLED               bool
DIVERTED                bool
IATA_ORIGIN         category
LATITUDE_ORIGIN      float64
LONGITUDE_ORIGIN     float64
IATA_DEST           category
LATITUDE_DEST        float64
LONGITUDE_DEST       float64
dtype: object